# Lab 03.1 – XADC Temperature Monitoring

## Objectives
- Identify the XADC device exposed by Linux through the Industrial I/O (IIO) subsystem.
- Read the raw on-chip temperature measurement.
- Use the IIO `raw`, `offset`, and `scale` values to calculate temperature.
- Create a reusable Python function for reading the Zynq die temperature.

## Hardware
- Digilent Zybo legacy board
- PYNQ 3.0.1
- No external sensor or additional hardware is required.

## Background
The Zynq-7000 device contains an internal XADC (Xilinx Analog-to-Digital Converter). In addition to external analog inputs, the XADC can monitor internal quantities such as the silicon die temperature and supply voltages.

On this PYNQ image, Linux exposes the XADC through the Industrial I/O (IIO) subsystem under `/sys/bus/iio/devices/`.

For the temperature channel, Linux provides three values:

- `raw` – raw ADC result
- `offset` – calibration offset
- `scale` – conversion scale

The physical value is calculated as:

\[
T_{m°C} = (raw + offset) \times scale
\]

The result is expressed in millidegrees Celsius, therefore:

\[
T_{°C} = \frac{T_{m°C}}{1000}
\]


## Exercise 1 – Locate the XADC device

The following code searches the available IIO devices and selects the device whose Linux name is `xadc`.


In [ ]:
from pathlib import Path

iio_root = Path("/sys/bus/iio/devices")
xadc_path = None

for dev in sorted(iio_root.glob("iio:device*")):
    name_file = dev / "name"
    if name_file.exists() and name_file.read_text().strip() == "xadc":
        xadc_path = dev
        break

if xadc_path is None:
    raise RuntimeError("XADC IIO device not found.")

print("XADC device:", xadc_path)
print("Device name:", (xadc_path / "name").read_text().strip())


List the files associated with the temperature channel.


In [ ]:
temperature_files = sorted(xadc_path.glob("in_temp*"))

for f in temperature_files:
    print(f.name)


## Exercise 2 – Read and convert the temperature

Read the raw ADC result together with the offset and scale supplied by the Linux IIO driver.


In [ ]:
raw = float((xadc_path / "in_temp0_raw").read_text().strip())
offset = float((xadc_path / "in_temp0_offset").read_text().strip())
scale = float((xadc_path / "in_temp0_scale").read_text().strip())

print(f"raw    = {raw}")
print(f"offset = {offset}")
print(f"scale  = {scale}")


Calculate the die temperature.


In [ ]:
temperature_mC = (raw + offset) * scale
temperature_C = temperature_mC / 1000.0

print(f"Zynq die temperature: {temperature_C:.2f} °C")


## Exercise 3 – Create a temperature-reading function

Encapsulate the conversion in a reusable Python function.


In [ ]:
def read_temperature_celsius(device=xadc_path):
    raw = float((device / "in_temp0_raw").read_text().strip())
    offset = float((device / "in_temp0_offset").read_text().strip())
    scale = float((device / "in_temp0_scale").read_text().strip())

    return (raw + offset) * scale / 1000.0

temperature = read_temperature_celsius()
print(f"Current temperature: {temperature:.2f} °C")


## Exercise 4 – Observe repeated measurements

Read the temperature several times. The values should be close, but they do not have to be identical.


In [ ]:
import time

for i in range(5):
    temperature = read_temperature_celsius()
    print(f"Measurement {i + 1}: {temperature:.2f} °C")
    time.sleep(1)


## Questions
1. What physical quantity does `in_temp0_raw` represent?
2. Why are `offset` and `scale` required instead of interpreting the raw value directly as degrees Celsius?
3. Is the measured temperature the ambient room temperature or the temperature of the Zynq silicon die?
4. Why can successive temperature measurements differ slightly?
5. What is the advantage of exposing XADC through the Linux IIO interface instead of accessing converter registers directly from Python?
